# Reproduce Main Results — *When Drift Breaks*

This notebook reproduces the main result table of

> *When Drift Breaks: Distributional Shape Deformation as a Crisis Detector*  \
> Plümer (2026), manuscript under preparation.

**Total runtime: ~30 seconds.**

The notebook loads the pre-computed seed-hardening results (10 random seeds × 3 phases) and prints the headline mean ± std table comparing pre-peak detection across four crises.

To run end-to-end from raw data instead, use `--mode full` (~3.5 hours of compute) — see the `README.md` for details.


In [ ]:
# Setup: ensure the repo is available
# This cell handles two situations:
#   (a) Notebook is opened from a cloned repo — nothing to do
#   (b) Notebook is opened in Colab without a clone — git-clone first
import os
import sys
from pathlib import Path

repo_root = Path.cwd()
# Heuristic: are we already inside the repo?
while not (repo_root / 'reproduce' / 'reproduce_main_results.py').exists():
    parent = repo_root.parent
    if parent == repo_root:
        # Reached filesystem root without finding the repo — clone it
        print('Cloning when-drift-breaks repository...')
        os.system('git clone https://github.com/lutzpluemer/when-drift-breaks.git /content/when-drift-breaks 2>&1 | tail -3')
        repo_root = Path('/content/when-drift-breaks')
        break
    repo_root = parent

os.chdir(repo_root)
print(f'Working directory: {repo_root}')
print(f'Repo contents: {sorted(p.name for p in repo_root.iterdir() if not p.name.startswith("."))}')


## Reproducing the headline table

The cell below invokes the reproduction script in *quick mode*: it loads the cached seed-hardening results from `data/seed_results/` and prints the mean ± std across 10 random seeds for each of the three phases.

The three phases are nested feature configurations:

| Phase                  | Features | Data sources                        |
|------------------------|----------|-------------------------------------|
| Shape                  | 57       | SPY                                 |
| Shape+Skew             | 60       | SPY + CBOE Skew Index               |
| Shape+Skew+Spectral    | 63       | SPY + Skew + 9 SPDR sector ETFs     |

Phase **Shape** captures distributional shape deformation and loss dynamics from SPY returns alone. **Shape+Skew** adds three columns derived from the CBOE Skew Index (implied tail risk). **Shape+Skew+Spectral** adds three columns of cross-sectional spectral structure (absorption ratio, leading-eigenvalue share, and its 10-day gradient) computed from a rolling correlation matrix of nine SPDR sector ETFs.


In [ ]:
# Reproduce the headline table from cached seed results
!python reproduce/reproduce_main_results.py --mode quick \
    --seed-results-dir data/seed_results


## How to read the table

Each cell shows **mean ± standard deviation** over 10 seeds (seed 42, 7, 13, 99, 123, 256, 512, 1024, 2048, 4096) of the particle filter, with the clustering layer fixed at `random_state=42`.

**Crisis pre-peak columns (China 2015, COVID 2020, Inflation 2022, Hormuz 2025):** Maximum stress score $P_t(\mathrm{Stress})$ in the 60 trading days *before* the crisis peak. Values close to 1.0 indicate strong pre-peak detection.

**TPR–FPR:** Difference between true-positive rate (within ±90 trading days of the four crisis peaks) and false-positive rate (in calm periods), at threshold 0.30. Larger is better. Monotonically improves from Shape (32.4%) to Shape+Skew+Spectral (41.6%) with non-overlapping confidence intervals.

**FPR 2017:** False-positive rate restricted to calendar year 2017, a benign year with no detected crisis. The two extended phases reach **0.0% across all 10 seeds**, providing a strong specificity guarantee.

### Reproducibility note

These results were computed using publicly available Yahoo Finance data (`data/yahoo/*.parquet`). The manuscript itself uses EODHD data; we verified that the spectral features driving the regime-detection pipeline reproduce between the two sources with correlations above 0.9999 and 95th-percentile absolute deviations below 0.001. See `README.md` for details.
